<div class="rqT", title="📥 README first">

[**Click here: Local (cloning) *vs*. Google Colab**](./colab_vs_local.md)

if you choose a Google Colab cloud session, click on this icon (unless *you are already* in Google Colab):

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rpoteau/pyPhysChem/blob/main/DS4B-CO2_solubility-ANN.ipynb)

</div>

In [ ]:
import requests

init_url = "https://raw.githubusercontent.com/rpoteau/pyPhysChem/main/colab_init.py"
try:
    r = requests.get(init_url, timeout=5)
    r.raise_for_status()
except requests.exceptions.RequestException:
    print("⚠️ Offline or connection error. Local setup assumed.")
else:
    exec(r.text)
    setup_environment(ML=True,clone_repo=False)

In [ ]:
import pyphyschemtools as t4pPC
from pyphyschemtools import fg, hl, bg, color
t4pPC.init()
%matplotlib inline

__author__ = "Romuald Poteau, LPCNO (INSA-CNRS-UT lab, Toulouse, France)"
print("pyphyschemtools, release",t4pPC.__version__)

# Prediction by an artificial neural network of the solubility of CO<sub>2</sub> in ionic liquids

<div class="introE">

<table width="100%" cellpadding="6"><tr><td bgcolor="#e6e6e6"><span style="color:#000000">📖 <b>Introduction</b></span></td></tr></table>
    
<b>Reference</b>: 
Z. Song, H. Shi, X. Zhang & T. Zhou (**2020**), Prediction of CO<sub>2</sub> solubility in ionic liquids using machine learning methods, [<i>Chem. Eng. Sci.</i> <b>223</b>: 115752](https://www.doi.org/10.1016/j.ces.2020.115752) 
<br>
<p style="text-align: center"><img width="650px" src="./DS4B-CO2-images/AbstractANNCO2-SongEtal.png" style="margin-left:auto; margin-right:auto" id="img_AbstractSong"></p>
<br>
The main results are graphically reported below.
<br>
<p style="text-align: center"><img width="900px" src="./DS4B-CO2-images/ANNCO2-SongEtal-Results.png" style="margin-left:auto; margin-right:auto" id="img_ResultsSong"></p>
<br>

**The goal of this tutorial is to reproduce the ANN part of this article, which is an example of application of a DFF neural network**

<img src="./DS4B-Slides/pngs/ZooNN/DFF-C.png" alt="DFF" width="180" style="display: block; margin: 0 auto; align:center" id="DFF"/>
</div>
<br>
<div class="rqT" title="A critical look">

Song <i>et al.</i>'s article is widely cited in the ionic-liquids/CO<sub>2</sub> capture literature, and its ANN has become something of a benchmark. Being widely cited is not the same as being methodologically sound, and part of the point of this tutorial is to go through the approach with a critical eye rather than taking "published and highly cited" as a guarantee. Two issues in particular will show up very concretely as we go:

- <b>Unconsolidated data.</b> The dataset aggregates many independent experimental sources without deduplication or cross-checking: as shown below, it contains hundreds of exactly duplicated rows, and dozens of (IL, T, P) conditions with two (or more) conflicting reported solubilities &mdash; a textbook case of data that looks clean until you actually check it.
- <b>No feature standardization.</b> The raw features mix binary group-contribution / one-hot descriptors (0 or 1) with *T* (hundreds of K) and *P* (up to several hundred bar) on wildly different scales. Feeding this directly into a gradient-based optimizer, as the first model below does, makes the learning collapse: the network essentially gives up on the chemistry and just predicts something close to the average solubility.

Reproducing the original approach without standardization is deliberate: these pitfalls are much easier to recognize in your own future work once you have actually seen what they look like in the numbers and in the plots, rather than only read about them as abstract warnings.

</div>

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib as mp
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
import os,sys
from IPython.display import display

<a id="data-read"></a>
## [A] Basic data analysis and visualization

<div class="introE">

<table width="100%" cellpadding="6"><tr><td bgcolor="#e6e6e6"><span style="color:#000000">📖 <b>Introduction</b></span></td></tr></table>

The [pandas library](https://pandas.pydata.org/docs/index.html) provides high-performance, easy-to-use data structures and data analysis tools for the Python programming language. It is recommended to first have a look at the [10mn short introduction to pandas](https://pandas.pydata.org/docs/user_guide/10min.html).
</div>

### [A.1] Read the database

<div class="introE">

<table width="100%" cellpadding="6"><tr><td bgcolor="#e6e6e6"><span style="color:#000000">📖 <b>Introduction</b></span></td></tr></table>

It is made of 57 columns:
- the name of the IL, under its abbreviated form. For example [BMIM][PF6] is for [1-Butyl-3-methylimidazolium hexafluorophosphate](https://en.wikipedia.org/wiki/1-Butyl-3-methylimidazolium_hexafluorophosphate)
- the abbreviated name of the cation and of the anion in two separated columns
- the CO<sub>2</sub> solubility (x_CO2)
- the temperature and pressure under which the solubility was measured 
- the 51 remaining columns are molecular descriptors. The authors chose:
    - group contribution (GC) descriptors for the cations, *i.e.* the occurrences of functional groups in the molecule
    - a one hot encoding scheme for anions, e.g. [1,0,...,0] (with 27 zeros) for [BF4], which is the first anion of this database, that contains ILs made of 28 different anions
</div>

In [ ]:
dataCO2f='DS4B-CO2-data'+'/'+'dataCO2.csv'
dataCO2=pd.read_csv(get_data(dataCO2f),sep=";",header=0)
display(dataCO2)
# describe() generates descriptive statistics
display(dataCO2.describe().style.format("{0:.2f}"))


In [ ]:
t4pPC.centerTitle("Unique ILs")
nIL = dataCO2["IL"].nunique()
print(f"There are {nIL} different ionic liquids in this database")
print(dataCO2["IL"].unique())

t4pPC.centerTitle("Unique cations")
nCat = dataCO2["cation"].nunique()
print(f"There are {nCat} different cations in this database")
print(dataCO2["cation"].unique())

t4pPC.centerTitle("Unique anions")
nAn = dataCO2["anion"].nunique()
print(f"There are {nAn} different anions in this database")
print(dataCO2["anion"].unique())

### [A.2] How do the cations an anions look like?

<div class="introE">

<table width="100%" cellpadding="6"><tr><td bgcolor="#e6e6e6"><span style="color:#000000">📖 <b>Introduction</b></span></td></tr></table>

Their SMILES coding is available in a separated csv file. `rdkit` will now be used to see the 2D structure of these species. The SMILES must first be converted into rdkit `mol` objects. 
</div>

In [ ]:
import rdkit
from rdkit.Chem import AllChem
from rdkit import Chem, DataStructs
from rdkit.Chem import PandasTools
from rdkit.Chem import Draw
from rdkit.Chem.Draw import IPythonConsole
print(f"{color.BOLD}{color.BLUE}Version of rdkit : {rdkit.__version__}{color.OFF}")
fSMILESILs='DS4B-CO2-data'+'/'+'dataCO2_SMILES_ILsList.csv'
SMILESILs=pd.read_csv(get_data(fSMILESILs),sep="\t",header=0)
display(SMILESILs)

In [ ]:
ILcation=SMILESILs[SMILESILs['Cation or Anion'] == 'Cation']
ILanion=SMILESILs[SMILESILs['Cation or Anion'] == 'Anion']
PandasTools.AddMoleculeColumnToFrame(ILcation, smilesCol='SMILES')
PandasTools.AddMoleculeColumnToFrame(ILanion, smilesCol='SMILES')
pd.options.display.max_rows = 5
display(ILcation)
display(ILanion)

#### [A.2.1] Display cations on a grid

In [ ]:
IPythonConsole.drawOptions.bondLineWidth=2
#FrameToGridImage: Draw grid image of mols in pandas DataFrame
imgCations = PandasTools.FrameToGridImage(ILcation, molsPerRow=6, legendsCol='Abbreviation', useSVG=True, subImgSize=(300, 200), maxMols=ILcation.shape[0])
display(imgCations)
t4pPC.save_text('DS4B-CO2-images/GridCations.svg', imgCations.data)

#### [A.2.2] Display anions on a grid

In [ ]:
IPythonConsole.drawOptions.bondLineWidth=2
#FrameToGridImage: Draw grid image of mols in pandas DataFrame
imgAnions = PandasTools.FrameToGridImage(ILanion, molsPerRow=6, legendsCol='Abbreviation', useSVG=True, subImgSize=(300, 200), maxMols=ILcation.shape[0])
display(imgAnions)
t4pPC.save_text('DS4B-CO2-images/GridAnions.svg', imgAnions.data)

### [A.3] Data distribution?

In [ ]:
t4pPC.chrono_start()
dataCO2.hist(figsize=(24,24))
print(dataCO2[["x_CO2", "[CH3]"]].groupby("[CH3]").count())
dataCO2.hist("[CH3]",figsize=(10,6),facecolor='g')
plt.show()
t4pPC.chrono_show()

## [B] Prediction of the solubility of CO<sub>2</sub> by an ANN: <br>a textbook case of work that does not quite meet the standards of the field 

### [B.1] ML scheme of the reference article

<div class="introE">

<table width="100%" cellpadding="6"><tr><td bgcolor="#e6e6e6"><span style="color:#000000">📖 <b>Introduction</b></span></td></tr></table>
    
<p style="text-align: center"><img width="800px" src="./DS4B-CO2-images/ANN-CO2Song_etal.png" style="margin-left:auto; margin-right:auto" id="img_Song"></p>

**Methodological details (excerpts from the article):**
- The MATLAB tansig and purelin transfer functions are employed in the hidden and output layers, respectively
- The experimental data are divided into a training set (8093 points, 80% of the data) to build the model and a test set of the remaining 2023 data points to evaluate the predictive capability of the obtained model
- <span style="color:red"><b>!!!</b> Instead of performing random selection, we employ a hybrid artificial-random strategy to decompose the dataset. Specifically, the data points consisting of the least frequently used groups are equally divided into five folders <b>!!!</b></span> 

</div>
<br>
<div class="rqT" title="Warning!">

This "*hybrid artificial-random strategy*" is not a valid strategy. The recommended K-fold cross validation, applied to the 3rd model, cannot be applied here!

**Let's observe now the weird behaviour of this model**
</div>

### [B.2] Any issue with the data?

<div class="rqT" title="Warning!">

Datasets compiled from the literature &mdash; like this one, built from many experimental papers &mdash; are prone to **duplicated entries**: the same experimental point can be copy-pasted twice, or reported independently by two different sources with a slightly different value.

This is not a cosmetic issue. A few cells below, the dataset is split into a training set and a test set with:

```python
data_train = dataCO2.sample(frac=0.8, axis=0)
data_test  = dataCO2.drop(data_train.index)
```

If the same (or a near-identical) experimental point appears twice in `dataCO2`, nothing prevents one copy from landing in `data_train` and the other in `data_test`. The ANN would then be partly evaluated on data it has already "seen" during training &mdash; a textbook case of **data leakage**, which makes the test MAE/R&sup2; look better than they really are.

**Let's check whether this dataset is affected, and clean it before going further.**

</div>

In [ ]:
t4pPC.centerTitle("Are there duplicated entries?")

# 1) Rows that are strictly identical on all 57 columns
n_exact = dataCO2.duplicated().sum()
print(f"Number of strictly identical rows (all columns): {n_exact}")

# 2) Rows reporting the same IL at the same (T,P): are they really independent measurements?
key_cols = ['IL','T (K)','P (bar)']
dup_mask = dataCO2.duplicated(subset=key_cols, keep=False)
print(f"Number of rows that share their (IL, T, P) with at least one other row: {dup_mask.sum()}")

display(dataCO2.loc[dup_mask, ['IL','T (K)','P (bar)','x_CO2']].sort_values(key_cols).head(10))

# 3) Among those, how many (IL,T,P) triplets report *different* x_CO2 values?
#    (= contradictory measurements, not simple copy-paste duplicates)
nunique_xco2 = dataCO2.groupby(key_cols)['x_CO2'].nunique()
conflicting_keys = nunique_xco2[nunique_xco2 > 1]
print(f"Number of (IL,T,P) triplets with inconsistent x_CO2 values: {len(conflicting_keys)}")

example_key = conflicting_keys.index[0]
mask_example = (
    (dataCO2['IL'] == example_key[0]) &
    (dataCO2['T (K)'] == example_key[1]) &
    (dataCO2['P (bar)'] == example_key[2])
)
display(dataCO2.loc[mask_example, ['IL','T (K)','P (bar)','x_CO2']])

### [B.3] Cleaning the dataset

<div class="introE">

<table width="100%" cellpadding="6"><tr><td bgcolor="#e6e6e6"><span style="color:#000000">📖 <b>Introduction</b></span></td></tr></table>

Two situations must be handled differently:
- **Exact duplicates** (same <code>x_CO2</code> value repeated for the same conditions): simply keep one occurrence.
- **Conflicting measurements** (same IL, same *T*, same *P*, but a different reported <code>x_CO2</code>, most probably coming from two different sources): rather than arbitrarily keeping one of them, their values are averaged. This is safe here because, for a given IL, all 51 molecular descriptor columns are identical by construction &mdash; only <code>x_CO2</code> needs to be aggregated.

The cleaned dataframe is stored in a new variable, <code>dataCO2_clean</code>, so that <code>dataCO2</code> is kept untouched for comparison.

</div>

In [ ]:
t4pPC.centerTitle("Removing duplicates and conflicting measurements")

# Step 1: remove strictly identical rows
dataCO2_clean = dataCO2.drop_duplicates(keep='first').copy()
print(f"{dataCO2.shape[0]} rows -> {dataCO2_clean.shape[0]} after removing exact duplicates")

# Step 2: for the (IL,T,P) triplets that still disagree on x_CO2,
# replace them with a single row carrying the averaged x_CO2
# (safe: the 51 descriptor columns only depend on the IL, not on T, P or x_CO2)
group_cols = [c for c in dataCO2_clean.columns if c != 'x_CO2']
dataCO2_clean = dataCO2_clean.groupby(group_cols, as_index=False)['x_CO2'].mean()
dataCO2_clean = dataCO2_clean[dataCO2.columns]   # restore the original column order

print(f"{dataCO2.shape[0]} rows -> {dataCO2_clean.shape[0]} after averaging the remaining conflicts")

assert dataCO2_clean.duplicated(subset=key_cols).sum() == 0
print("No duplicated (IL, T, P) entries left in dataCO2_clean.")

display(dataCO2_clean.describe().style.format("{0:.2f}"))

<div class="rqE">

<table width="100%" cellpadding="6"><tr><td bgcolor="#fde9e9"><span style="color:#B71C1C">⚠️ <b>Pay Attention</b></span></td></tr></table>

From this point on, it would be more rigorous to replace <code>dataCO2</code> by <code>dataCO2_clean</code> everywhere a train/test split is performed below. This is left on purpose: keep it in mind while progressing through the models, and come back to this cell if you want to measure how much this cleaning step changes the reported MAE/R&sup2; values.

</div>

In [ ]:
dataCO2 = dataCO2_clean

### [B.4] Useful function

In [ ]:
def plotModel(model, model_id, MODELS_PATH=".", fig_extension="png", resolution=300,
              show_shapes=True, show_layer_names=True, show_layer_activations=True):
    '''
    saves a model plot to a given subfolder, that will be created if it does not exist
    input:
        - model = keras model
        - model_id = name of the model file, without an extension
        - MODELS_PATH = pathway to the folder. Default: "."
        - model_extension = model type. Default: png
    '''
    from tensorflow.keras.utils import plot_model
    from matplotlib.pyplot import plot
    path = MODELS_PATH / f"{model_id}.{fig_extension}"
    plot_model(model, to_file=path, dpi=resolution, show_shapes=show_shapes, 
               show_layer_names=show_layer_names, show_layer_activations=show_layer_activations)
    img = plt.imread(path)
    plt.imshow(img)
    plt.axis("off")
    plt.show()
    return


### [B.5] Model 1: assesment of the reproducibility of the simulation of Song *et al.*<a id="model1"></a>

<div class = "introE">

<table width="100%" cellpadding="6"><tr><td bgcolor="#e6e6e6"><span style="color:#000000">📖 <b>Introduction</b></span></td></tr></table>
    
Errors calculated between the actual and predicted x_CO2 values (noted $Y$ and $\hat{Y}$ in equations below)  
MAE =  Mean Absolute Error 
$$\mathrm{MAE}=\frac{1}{n}\sum_{i}\left|\hat{Y}(x_{i})-Y(x_{i})\right|$$

MSE = Mean Squared Error  
$$\mathrm{MSE}=\frac{1}{n}\sum_{i}\left(\hat{Y}(x_{i})-Y(x_{i})\right)^{2}$$

</div>

#### [B.5.1] Split the data into train and test sets

In [ ]:
#######################################################################################
# separation of the data set into two subsets: (1) training of the ANN & (2) test of the ANN
# library used: pandas
data_train = dataCO2.sample(frac=0.8, axis=0)
data_test  = dataCO2.drop(data_train.index)

x_train = data_train.drop(['IL','cation','anion','x_CO2'],axis=1)
y_train = data_train['x_CO2']
x_test  = data_test.drop(['IL','cation','anion','x_CO2'],axis=1)
y_test  = data_test['x_CO2']

display(x_train,y_train)
display(x_test,y_test)

#### [B.5.2] Define the ANN

In [ ]:
#######################################################################################
# ANN: 1 input layer (53 neurons) / 1 hidden layer (7 neurons with the reLu activation function) / 1 output layer (1 neuron with no activation function) 
# library used: keras

def defANN(shape,acthL):
    
    model = keras.models.Sequential()
    model.add(keras.layers.Input(shape, name='iLayer'))
    model.add(keras.layers.Dense(7, activation=acthL, name='hLayer'))
    model.add(keras.layers.Dense(1, name='oLayer'))
    
    model.compile(optimizer = 'adam',
                  loss      = 'mse',
                  metrics   = ['mae', 'mse', keras.metrics.R2Score()] )
    return model

acthL='tanh'
#acthL='relu'
if 'ANNmodel' in globals():
    del ANNmodel
if 'ANNhistory' in globals():
    del ANNhistory
ANNmodel=defANN( (53,), acthL )
ANNmodel.summary()

#### [B.5.3] Plot the model

In [ ]:
from pathlib import Path

MODELS_PATH = Path() / "DS4B-CO2-SavedModels"
MODELS_PATH.mkdir(parents=True, exist_ok=True)
plotModel(ANNmodel, "Model1_CO2", MODELS_PATH=MODELS_PATH)

#### [B.5.4] Train the ANN

In [ ]:
#######################################################################################
# optimization of the ANN
# library used: keras

from tensorflow.keras.callbacks import EarlyStopping
es = EarlyStopping(monitor='val_loss', mode='min', verbose=1, patience=15)
t4pPC.chrono_start()
ANNhistory = ANNmodel.fit(x_train,
                    y_train,
                    epochs          = 300,
                    batch_size      = 20,
                    verbose         = 1,
                    validation_data = (x_test, y_test),
                    callbacks=[es])
t4pPC.chrono_show()

#######################################################################################

<div class="rqE">

<table width="100%" cellpadding="6"><tr><td bgcolor="#fde9e9"><span style="color:#B71C1C">⚠️ <b>Pay Attention</b></span></td></tr></table>
<span style="color:red; font-weight:bold">The MAE and R2 parameters seem rather good (<i>ca.</i> 0.07 and 0.92), but this prediction is actually very bad, as you will see by jumping to the <a href="#model-evaluation">Model Evaluation</a> section!</span>
<br>
<br>  
Unless you were lucky (an element of chance is involved in the learning process), the predicted <i>vs.</i> actual CO<sub>2</sub> solubility probably looks like this: 
<br>
<p style="text-align: center"><img width="550px" src="./DS4B-CO2-images/predictionCO2_woStd-saved.png" style="margin-left:auto; margin-right:auto" id="img_predCO2woStd"></p>
<b>It differs a lot from the results of the reference article, reported in the introduction of this notebook.</b>
<br><br>
<span style="color:red; font-weight:bold">Origin of this weird behaviour?</span>
<br>
Probably the absence of <b>standardization</b> of the data. This scattering is by the way reminiscent the variation of the solubility as a function of <i>T</i> or <i>p</i>, plotted below. Let's check that in the next script.
</div>

In [ ]:
cm = 1/2.54
plt.rcParams["figure.figsize"] = (20*cm,12*cm) #graphic size
fig1=dataCO2.plot(x="T (K)", y=["x_CO2"],linestyle='', marker='o',fontsize=14)
fig2=dataCO2.plot(x="P (bar)", y=["x_CO2"],linestyle='', marker='x',fontsize=14)

<a id="model2"></a>
## [C] Model 2 = standardization of *T* and *p* only

<div class="rqE">

<table width="100%" cellpadding="6"><tr><td bgcolor="#fde9e9"><span style="color:#B71C1C">⚠️ <b>Pay Attention</b></span></td></tr></table>
    
**What is standardization?**

**Standardization** is the process of translating and scaling our features so that they are all *distributed around a mean of zero* with a *standard deviation of one*.
    
Because the Temperature and pressure in the CO<sub>2</sub> dataset are on totally different scales, we need to scale them, **without changing their relative distribution**
    
We want to standardize our data so that the *covariances* are easily comparable for each pair of features. If we don’t do it, features with larger ranges of numbers will have higher covariances, which could be an artificial feature.
</div>

<br>
<div class="introE">

<table width="100%" cellpadding="6"><tr><td bgcolor="#e6e6e6"><span style="color:#000000">📖 <b>Introduction</b></span></td></tr></table>

This new model  is an improvement of model 1, but without standardizing the GC descriptors which covariances are small. <br>
</div>
<br>

### [C.1] Split into train and test sets

In [ ]:
# Resets all state generated by Keras
tf.keras.backend.clear_session()

In [ ]:
#######################################################################################
# separation of the data set into two subsets: (1) training of the ANN & (2) test of the ANN
# library used: pandas
data_train = dataCO2.sample(frac=0.8, axis=0)
data_test  = dataCO2.drop(data_train.index)

x_train = data_train.drop(['IL','cation','anion','x_CO2'],axis=1)
y_train = data_train['x_CO2']
x_test  = data_test.drop(['IL','cation','anion','x_CO2'],axis=1)
y_test  = data_test['x_CO2']

#######################################################################################

# Train
x_train = data_train.drop(['IL','cation','anion','x_CO2'],axis=1)
xIL_train = x_train.drop(['T (K)','P (bar)'],axis=1)
xTP_train = data_train[['T (K)','P (bar)']]
y_train = data_train['x_CO2']
x_trainTmp = x_train.copy() # for plotting purpose

# Test
x_test  = data_test.drop(['IL','cation','anion','x_CO2'],axis=1)
xIL_test = x_test.drop(['T (K)','P (bar)'],axis=1)
xTP_test = data_test[['T (K)','P (bar)']]
y_test  = data_test['x_CO2']
x_testTmp = x_test.copy() # for plotting purpose

print(f"{bg.DARKREDB}Training set before normalization:{bg.OFF}")
display(x_train)
display(x_train.describe().style.format("{0:.2f}").set_caption("Training set before normalization:"))
print(f"{bg.DARKREDB}Test set before normalization:{bg.OFF}")
display(x_test)
display(x_test.describe().style.format("{0:.2f}").set_caption("Test set before normalization:"))

### [C.2] Scale independently T & P

In [ ]:
# using scikit-learn
from sklearn import preprocessing

print(f"{bg.LIGHTBLUEB}Training set (T,P) after normalization:{bg.OFF}")
scalerTP = preprocessing.StandardScaler()
scalerTP.fit(xTP_train.values)
xTP_train_scaled = scalerTP.transform(xTP_train.values) #returns a numpy array
xTP_train = pd.DataFrame(xTP_train_scaled, index=xTP_train.index, columns=xTP_train.columns)
display(xTP_train)
display(xTP_train.describe().style.format("{0:.2f}"))

print(f"{bg.LIGHTBLUEB}Test set (T,P) after normalization:{bg.OFF}")
xTP_test_scaled = scalerTP.transform(xTP_test.values) #returns a numpy array
xTP_test = pd.DataFrame(xTP_test_scaled, index=xTP_test.index, columns=xTP_test.columns)
display(xTP_test)
display(xTP_test.describe().style.format("{0:.2f}"))


### [C.3] Merge the (xTP_train & xIL_train) dfs & (xTP_test & xIL_test) dfs

In [ ]:
x_train = pd.concat([xTP_train,xIL_train], axis=1)
x_test = pd.concat([xTP_test,xIL_test], axis=1)
display(x_train.describe().style.format("{0:.2f}"))
display(x_test.describe().style.format("{0:.2f}"))

In [ ]:
#######################################################################################
# graphic representation of the previous tables
import matplotlib.gridspec as gridspec
from matplotlib.ticker import MaxNLocator

nbins=30
fig = plt.figure(figsize=(18, 12))
plt.rcParams["font.size"] = (14) #font size
plt.tick_params(axis='x',labelsize = 16)
plt.tick_params(axis='y',labelsize = 12)
gs = gridspec.GridSpec(2, 4, height_ratios=[1, 1], width_ratios=[1, 1, 1, 1])

ax00=plt.subplot(gs[0,0], title="Tr. set before standardization")
fig00=sns.histplot(data=x_trainTmp,x="T (K)", bins=nbins, kde=True, color="#1f77b4", alpha=1.0)
ax02=plt.subplot(gs[0,2], title="Tr. set before standardization")
fig02=sns.histplot(data=x_trainTmp,x="P (bar)", bins=nbins, kde=True, color="#3e95ff", alpha=1.0)
ax01=plt.subplot(gs[0,1], title="Tr. set after standardization")
fig01=sns.histplot(data=x_train,x="T (K)", bins=nbins, kde=True, color="#1f77b4", alpha=1.0)
ax03=plt.subplot(gs[0,3], title="Tr. set after standardization")
fig11=sns.histplot(data=x_train,x="P (bar)", bins=nbins, kde=True, color="#3e95ff", alpha=1.0)

ax10=plt.subplot(gs[1,0], title="Ts. set before standardization")
fig10=sns.histplot(data=x_testTmp,x="T (K)", bins=nbins, kde=True, color="#1f77b4", alpha=1.0)
ax12=plt.subplot(gs[1,2], title="Ts. set before standardization")
fig12=sns.histplot(data=x_testTmp,x="P (bar)", bins=nbins, kde=True, color="#3e95ff", alpha=1.0)
ax11=plt.subplot(gs[1,1], title="Ts. set after standardization")
fig11=sns.histplot(data=x_test,x="T (K)", bins=nbins, kde=True, color="#1f77b4", alpha=1.0)
ax13=plt.subplot(gs[1,3], title="Ts. set after standardization")
fig13=sns.histplot(data=x_test,x="P (bar)", bins=nbins, kde=True, color="#3e95ff", alpha=1.0)


plt.savefig('./DS4B-CO2-images/Standardization_CO2data.png',dpi=300,bbox_inches='tight')

### [C.4] Delete the scaled & Tmp dfs

In [ ]:
del x_testTmp, x_trainTmp, xTP_test_scaled, xTP_train_scaled

### [C.5] Now define and train the ANN

In [ ]:
#######################################################################################
# ANN: 1 input layer (53 neurons) / 2 hidden layers (20 and 7 neurons) / 1 output layer (1 neuron) 
# library used: keras

def defANN(shape,acthL):
    
    model = keras.models.Sequential()
    model.add(keras.layers.Input(shape, name='iLayer'))
    model.add(keras.layers.Dense(7, activation=acthL, name='hLayer'))
    model.add(keras.layers.Dense(1, name='oLayer'))
    
    model.compile(optimizer = 'adam',
                  loss      = 'mse',
                  metrics   = ['mae', 'mse', keras.metrics.R2Score()] )
    return model

acthL='tanh'
#acthL='relu'
if 'ANNmodel' in globals():
    del ANNmodel
if 'ANNhistory' in globals():
    del ANNhistory
ANNmodel=defANN( (53,), acthL )
ANNmodel.summary()

In [ ]:
#######################################################################################
# optimization of the ANN
# library used: keras

from tensorflow.keras.callbacks import EarlyStopping
es = EarlyStopping(monitor='val_loss', mode='min', verbose=1, patience=15)

t4pPC.chrono_start()
ANNhistory = ANNmodel.fit(x_train,
                    y_train,
                    epochs          = 300,
                    batch_size      = 20,
                    verbose         = 1,
                    validation_data = (x_test, y_test),
                    callbacks=[es])
t4pPC.chrono_show()

#######################################################################################

<div class="rqE">

<table width="100%" cellpadding="6"><tr><td bgcolor="#fde9e9"><span style="color:#B71C1C">⚠️ <b>Pay Attention</b></span></td></tr></table>

Once completed, jump to the <a href="#model-evaluation">Model Evaluation</a> section

</div>

## [D] Assessment of the stability of the ML model 2 by *K*-fold cross validation<a id="model4"></a>

<div class="rqE">

<table width="100%" cellpadding="6"><tr><td bgcolor="#fde9e9"><span style="color:#B71C1C">⚠️ <b>Pay Attention</b></span></td></tr></table>
Model 2 is now adapted to assess how the results of the ML generalizes</div>
<br>

<div class="introE">

<table width="100%" cellpadding="6"><tr><td bgcolor="#e6e6e6"><span style="color:#000000">📖 <b>Introduction</b></span></td></tr></table>
The main idea behind cross-validation is that each sample in a dataset has the opportunity of being tested. K-fold cross-validation is a special case of cross-validation where we iterate over a dataset set <i>k</i> times. The dataset is split into <i>k</i> parts at each iteration: one part is used for validation, and the remaining <i>k</i>−1  parts are merged into a training subset for model evaluation. The figure below illustrates the process of 5-fold cross-validation
<p style="text-align: center"><img width="520px" src="DS4B-svg/KFoldCV.svg" style="margin-left:auto; margin-right:auto" id="img_5FoldCV"></p>
</div>

### [D.1] Apply a 5-fold cross validation

In [ ]:
#######################################################################################
# separation of the data set into two subsets: (1) training of the ANN & (2) test of the ANN
# library used: pandas
xdata = dataCO2.drop(['IL','cation','anion','x_CO2'],axis=1)
xdataIL = xdata.drop(['T (K)','P (bar)'],axis=1)
xdataTP = xdata[['T (K)','P (bar)']]
ydata = dataCO2['x_CO2']

#######################################################################################
# ANN: 1 input layer (53 neurons) / 2 hidden layers (20 and 7 neurons) / 1 output layer (1 neuron) 
# library used: keras

def defANN(shape,acthL):
    
    model = keras.models.Sequential()
    model.add(keras.layers.Input(shape, name='iLayer'))
    model.add(keras.layers.Dense(7, activation=acthL, name='hLayer'))
    model.add(keras.layers.Dense(1, name='oLayer'))
    
    model.compile(optimizer = 'adam',
                  loss      = 'mse',
                  metrics   = ['mae', 'mse', keras.metrics.R2Score()] )
    return model

acthL='tanh'

In [ ]:
#######################################################################################
# optimization of the ANN
# library used: keras and scikit learn for the KFold cross-validator
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import KFold
from sklearn import preprocessing
from pickle import dump
from sklearn.metrics import r2_score

VERBOSE = 1
EPOCHS = 300
BATCH_SIZE = 25
N_SPLIT = 5

t4pPC.chrono_start()

# variables created to save at each iteration of the KFold process: the mean error, the standard deviation, MAE, R2
meantT=list()
stdtT=list()
MAEtT=list()
R2tT=list()
meanvT=list()
stdvT=list()
MAEvT=list()
R2vT=list()

kfold = KFold(n_splits=N_SPLIT,shuffle=True,random_state=42) # k-fold is here!
#print(list(kfold.split(x_train,y_train)))

historiesT = list()   # to keep every fold's History object
j = 0 # Variable for keeping count of split we are executing
# The KFold cv provides train/test indices to split data in train/test sets
for train_idx, val_idx in list(kfold.split(xdata,ydata)):

    tf.keras.backend.clear_session()   # reset Keras state before building a new model
    # (also resets the automatic layer-naming counters, so model.summary()
    #  looks the same — e.g. 'dense', 'dense_1' — at every fold)
    es = EarlyStopping(monitor='val_loss', mode='min', verbose=1, patience=15)   # fresh instance, no state carried over from the previous fold

    xIL_train_cv = xdataIL.iloc[train_idx]
    xIL_valid_cv = xdataIL.iloc[val_idx]
    xTP_train_cv = xdataTP.iloc[train_idx]
    xTP_valid_cv = xdataTP.iloc[val_idx]
    y_train_cv = ydata.iloc[train_idx]
    y_valid_cv = ydata.iloc[val_idx]
#    display(x_train_cv,x_valid_cv)
    scalerTP = preprocessing.StandardScaler()
    scalerTP.fit(xTP_train_cv.values)
    xTPt_scaled = scalerTP.transform(xTP_train_cv.values) #returns a numpy array
    xTPv_scaled = scalerTP.transform(xTP_valid_cv.values) #returns a numpy array
    xtmp_train_cv = pd.DataFrame(xTPt_scaled, index=xTP_train_cv.index, columns=xTP_train_cv.columns) #conversion of a numpy array into a pandas dataframe
    xtmp_valid_cv = pd.DataFrame(xTPv_scaled, index=xTP_valid_cv.index, columns=xTP_valid_cv.columns) #conversion of a numpy array into a pandas dataframe
    x_train_cv = pd.concat([xtmp_train_cv,xIL_train_cv], axis=1)
    x_valid_cv = pd.concat([xtmp_valid_cv,xIL_valid_cv], axis=1)
    del xTPt_scaled, xTPv_scaled, xtmp_train_cv, xtmp_valid_cv, xIL_train_cv, xIL_valid_cv
    del xTP_train_cv, xTP_valid_cv
#    display(x_train_cv.describe().style.format("{0:.2f}").set_caption("Training set after normalization (with scikit-learn):"))
#    display(x_valid_cv.describe().style.format("{0:.2f}").set_caption("Validation set after normalization (with scikit-learn):"))
    print(f"{color.BOLD}{color.RED}Fold {j}{color.OFF}")
    j+=1
    ANNmodel=defANN( (53,), acthL )
    ANNhistory = ANNmodel.fit(x_train_cv,
                        y_train_cv,
                        epochs          = EPOCHS,
                        batch_size      = BATCH_SIZE,
                        verbose         = VERBOSE,
                        validation_data = (x_valid_cv, y_valid_cv),
                        callbacks=[es])
    historiesT.append(ANNhistory)   # store the whole History object, not just .history
    ytrain_hat=ANNmodel.predict(x_train_cv)
    yvalid_hat=ANNmodel.predict(x_valid_cv)
    diffyt = ytrain_hat.flatten() - y_train_cv.to_numpy()
    diffyv = yvalid_hat.flatten() - y_valid_cv.to_numpy()

    print()
    print("xCO2(predicted) - xCO2(actual)")
    print(f"Train.  mean: {np.mean(diffyt):.3f}"
          f"   std: {np.std(diffyt):.3f}"
          f"   MAE: {np.average(abs(diffyt)):.3f}"
          f"    R2: {r2_score(y_train_cv.to_numpy(), ytrain_hat.flatten()):.3f}")
    print(f"Test.   mean: {np.mean(diffyv):.3f}"
          f"   std: {np.std(diffyv):.3f}"
          f"   MAE: {np.average(abs(diffyv)):.3f}"
          f"    R2: {r2_score(y_valid_cv.to_numpy(), yvalid_hat.flatten()):.3f}")
    meantT.append(np.mean(diffyt))
    meanvT.append(np.mean(diffyv))
    stdtT.append(np.std(diffyt))
    stdvT.append(np.std(diffyv))
    MAEtT.append(np.average(abs(diffyt)))
    MAEvT.append(np.average(abs(diffyv)))
    R2tT.append(r2_score(y_train_cv.to_numpy(), ytrain_hat.flatten()))
    R2vT.append(r2_score(y_valid_cv.to_numpy(), yvalid_hat.flatten()))
    
    # Save the Keras ANN model for later use    
    ModelName = './DS4B-CO2-SavedModels/ANNmodel-CO2-StandardizedB-' + str(j-1) + '.keras'
    ANNmodel.save(ModelName)
    # Scaler also saved!!!!!!!!!!!!!!
    ScalerName = './DS4B-CO2-SavedModels/scalerB-' + str(j-1) + '.pkl'
    dump(scalerTP, open(ScalerName, 'wb'))
    
t4pPC.chrono_show()

#######################################################################################
# accuracy of the ANN?
# library used: numpy
print(f"{color.BOLD}average MAE of the training set:{color.OFF}   {np.mean(MAEtT):.2f} +/- {np.std(MAEtT):.2f}")
print(f"{color.BOLD}average MAE of the validation set:{color.OFF} {np.mean(MAEvT):.2f} +/- {np.std(MAEvT):.2f}")

### [D.2] Visual assessment of the 5-fold cross validation

In [ ]:
figCV, axCV = plt.subplots(1, 1)
figCV.set_size_inches(12,5)
axCV.errorbar(x=np.arange(len(meantT)), y=meantT, yerr=MAEtT, label='training sets', fmt='o-', capsize=10)
axCV.errorbar(x=np.arange(len(meanvT))+0.1, y=meanvT, yerr=MAEvT, label='validation sets', fmt='o-', capsize=10)
axCV.legend(loc='lower left', shadow=True, fontsize='14')
axCV.set_xlabel('cross-validation k-values ',fontdict={'fontsize':16})
axCV.set_ylabel('$\hat{y}-y_{\mathrm{actual}}$',fontdict={'fontsize':16})
axCV.tick_params(labelsize = 14)
plt.savefig('./DS4B-CO2-images/KFold-cv.png',dpi=300,bbox_inches='tight')
plt.show()

<div class="rqE">

<table width="100%" cellpadding="6"><tr><td bgcolor="#fde9e9"><span style="color:#B71C1C">⚠️ <b>Pay Attention</b></span></td></tr></table>
Cross-validation is a resampling procedure used to evaluate machine learning models on a limited data sample. It generally results in a less biased or less optimistic estimate of the model skill than other methods, such as a simple train/test split. Although the CO<sub>2</sub> data set contains more than 10k samples, it is always worth proving the stability of an ML algorithm. As a result, it is a pity that a scheme such as the one just determined above (and in principle close to the one reported below) is not available in the Song <i>et al.</i> publication, at least in the supplementary information part.
<p style="text-align: center"><img width="650px" src="./DS4B-CO2-images/KFold-cvK.png" style="margin-left:auto; margin-right:auto" id="img_ResultsSong"></p>
<b>This error plot shows a good performance of the ML algorithm, as well as similar error bars, regardless of the training data set.</b>
</div>

<div class="rqE">

<table width="100%" cellpadding="6"><tr><td bgcolor="#fde9e9"><span style="color:#B71C1C">⚠️ <b>Pay Attention</b></span></td></tr></table>

**Do you want to evaluate the last optimized model of the K-fold process?**

evaluate the next Python cell for compatibility purpose with the **Model Evaluation** section
</div>

In [ ]:
#######################################################################################
# pick the best fold (highest validation R2) instead of just keeping the last one
# library used: numpy, pickle, tensorflow.keras

best_fold = int(np.argmax(R2vT))   # fold with the highest validation R2
print(f"Best fold: {best_fold}  (validation R2 = {R2vT[best_fold]:.3f})")

from pickle import load
from tensorflow.keras.models import load_model

# reload the model and the scaler that were saved for that specific fold
ModelName  = './DS4B-CO2-SavedModels/ANNmodel-CO2-StandardizedB-' + str(best_fold) + '.keras'
ScalerName = './DS4B-CO2-SavedModels/scalerB-' + str(best_fold) + '.pkl'
ANNmodel  = load_model(ModelName)
scalerTP  = load(open(ScalerName, 'rb'))
ANNhistory = historiesT[best_fold]

# re-generate the exact train/validation split used for that fold
# (deterministic: same random_state=42, so splitting again reproduces it exactly)
train_idx, val_idx = list(kfold.split(xdata, ydata))[best_fold]

xIL_train_cv = xdataIL.iloc[train_idx]
xIL_valid_cv = xdataIL.iloc[val_idx]
xTP_train_cv = xdataTP.iloc[train_idx]
xTP_valid_cv = xdataTP.iloc[val_idx]
y_train_cv   = ydata.iloc[train_idx]
y_valid_cv   = ydata.iloc[val_idx]

xTPt_scaled   = scalerTP.transform(xTP_train_cv.values)
xTPv_scaled   = scalerTP.transform(xTP_valid_cv.values)
xtmp_train_cv = pd.DataFrame(xTPt_scaled, index=xTP_train_cv.index, columns=xTP_train_cv.columns)
xtmp_valid_cv = pd.DataFrame(xTPv_scaled, index=xTP_valid_cv.index, columns=xTP_valid_cv.columns)
x_train_cv = pd.concat([xtmp_train_cv, xIL_train_cv], axis=1)
x_valid_cv = pd.concat([xtmp_valid_cv, xIL_valid_cv], axis=1)

ytrain_hat = ANNmodel.predict(x_train_cv)
yvalid_hat = ANNmodel.predict(x_valid_cv)

# overwrite x_train/x_test/y_train/y_test so everything downstream (evaluate, SHAP, ...)
# uses the SAME best-fold data consistently, instead of mixing the original split's x
# with the last fold's y
x_train    = x_train_cv.copy()
x_test     = x_valid_cv.copy()
y_train    = y_train_cv.copy()
y_test     = y_valid_cv.copy()
ytest_hat  = yvalid_hat.copy()


<a id="model-evaluation"></a>
## [E] Model evaluation

### [E.1] Loss, MAE and MSE as a function of epochs = during the ANN optimization with the back-propagation algorithm 

In [ ]:
ANNscoreTrain= ANNmodel.evaluate(x_train, y_train, verbose=0)
print(f"{color.GREEN}x_train / loss      : {ANNscoreTrain[0]:5.4f}")
print(f"x_train / mae       : {ANNscoreTrain[1]:5.4f}")
print(f"x_train / mse       : {ANNscoreTrain[2]:5.4f}")
print(f"x_train / R2        : {ANNscoreTrain[3]:5.3f}{color.OFF}")
print()
ANNscoreEval = ANNmodel.evaluate(x_test, y_test, verbose=0)
print(f"{color.BLUE}x_test / loss       : {ANNscoreEval[0]:5.4f}")
print(f"x_test / mae        : {ANNscoreEval[1]:5.4f}")
print(f"x_test / mse        : {ANNscoreEval[2]:5.4f}")
print(f"x_test / R2         : {ANNscoreEval[3]:5.3f}{color.OFF}")

df = pd.DataFrame(data=ANNhistory.history)

metrics = [
    ("mae",      "mae", (0.0, 0.2)),
    ("mse",      "mse", (0.0, 0.1)),
    ("r2_score", "R2",  None),   # pas fixed ylim : R2 can be negative
]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (key, label, ylim) in zip(axes, metrics):
    ax.plot(df[key],          linestyle='-', marker='o', label='train')
    ax.plot(df[f"val_{key}"], linestyle='-', marker='o', label='validation')
    ax.set_xlabel('epoch', fontsize=14)
    ax.set_ylabel(label, fontsize=14)
    ax.tick_params(labelsize=12)
    if ylim is not None:
        ax.set_ylim(ylim)
    ax.legend(loc='best', shadow=True, fontsize=12)

plt.tight_layout()
plt.show()

### [E.2] Comparison between experimental and predicted CO<sub>2</sub> solubility

In [ ]:
import matplotlib.gridspec as gridspec
from matplotlib.ticker import MaxNLocator

ytrain_hat=ANNmodel.predict(x_train)
ytest_hat=ANNmodel.predict(x_test)
diffyt = ytrain_hat.flatten() - y_train.to_numpy()
diffyp = ytest_hat.flatten() - y_test.to_numpy()

xD=[0,1]
yD=[0,1]
fig = plt.figure(figsize=(8, 12))
plt.rcParams["font.size"] = (18) #font size
gs = gridspec.GridSpec(2, 1, height_ratios=[10, 4], hspace=0, figure=fig)
ax0=fig.add_subplot(gs[0,0])
ax0.plot(y_train,ytrain_hat,'o',color="#1f77b4",label='training set')
ax0.plot(y_test,ytest_hat,'P',color="orange",label='validation set')
ax0.plot(xD,yD,'c',linestyle='--',lw=3)
ax0.set_xlabel('experimental CO$_2$ solubility',fontsize=16)
ax0.set_ylabel('predicted CO$_2$ solubility',fontsize=16)
ax0.set_xlim(0,1);ax0.set_ylim(0,1) 
ax0.legend(loc='upper left', shadow=True, fontsize='16')
plt.setp(ax0.get_xticklabels(), visible=False) #cosmetics

ax1=fig.add_subplot(gs[1,0])
ax1.plot(y_train,diffyt,'o',color="#1f77b4",label='training set')
ax1.plot(y_test,diffyp,'P',color="orange",label='validation set')
ax1.axhline(y=0, xmin=0., xmax=1.0, color='c', linestyle='--', linewidth=3)
ax1.set_xlabel('experimental CO$_2$ solubility',fontsize=16)
ax1.set_ylabel('residuals (exp./pred. error)',fontsize=16)
ax1.set_xlim(0.0,1.0); ax1.set_ylim(-0.3,0.3)
ax1.axes.get_yaxis().set_major_locator(MaxNLocator(prune='upper'))  #cosmetics
gs.tight_layout(fig)
plt.tick_params(labelsize = 14)
plt.savefig('./DS4B-CO2-images/predictionCO2.png',dpi=300,bbox_inches='tight')
plt.show()


### [E.3] Residuals plotted as histograms (distribution of the error)

In [ ]:
from sklearn.metrics import r2_score
print(f"{color.BLUE}"
      f"Train. mean: {np.mean(diffyt):.3f}"
      f"   std: {np.std(diffyt):.3f}"
      f"   MAE: {np.average(abs(diffyt)):.3f}"
      f"    R2: {r2_score(y_train.to_numpy(), ytrain_hat.flatten()):.3f}")

print(f"{color.YELLOW}"
      f" Test. mean: {np.mean(diffyp):.3f}"
      f"   std: {np.std(diffyp):.3f}"
      f"   MAE: {np.average(abs(diffyp)):.3f}"
      f"    R2: {r2_score(y_test.to_numpy(), ytest_hat.flatten()):.3f}")

# make a dataframe
df1 = pd.DataFrame(
    {
        "Error distribution": diffyp,
        "Type": 'Test set'    },
)
#display(df1)
df2 = pd.DataFrame(
    {
        "Error distribution": diffyt,
        "Type": 'Training set'    },
)
#display(df2)
distrib=pd.concat([df1,df2],ignore_index=True)
#display(distrib)

xlim=0.2
nbins=120

import matplotlib.gridspec as gridspec
plt.rcParams["figure.figsize"] = (8,13) #graphic size
plt.rcParams["font.size"] = (18) #font size
gs = gridspec.GridSpec(2, 1, height_ratios=[3, 1])
plt.subplot(gs[0])
fig1=sns.histplot(data=distrib,x="Error distribution", hue="Type", bins=nbins, kde=True, palette=['Orange','#1f77b4'], alpha=1.0)
plt.xlabel("$x_{\mathrm{CO_2}}^{\mathrm{pred}}-x_{\mathrm{CO_2}}^{\mathrm{exp}}$")
plt.ylabel("number of data points in each interval")
plt.xlim(-xlim,xlim)
plt.subplot(gs[1])
plt.xlim(-xlim,xlim)
fig2=sns.boxplot(data=distrib, x="Error distribution", hue="Type", notch=True, orient = 'h',palette=['Orange','#1f77b4'],legend=False)
fig2.set(ylabel=None)
plt.savefig('./DS4B-CO2-images/errorCO2.png',dpi=300,bbox_inches='tight')

### [E.4] Identify the most significant failures?

<div class="introE">

<table width="100%" cellpadding="6"><tr><td bgcolor="#e6e6e6"><span style="color:#000000">📖 <b>Introduction</b></span></td></tr></table>

Looking only at global metrics (MAE, R²) hides *where* the model actually struggles. The following cells dig into that, step by step:

1. **Collect the predictions**: merge the training and test predictions (<code>x_CO2_hat</code>) back into a single dataframe aligned with the original data, <code>dataCO2_hat</code>.
2. **Quantify the error per point**: compute the residual (<code>x_CO2_hat − x_CO2</code>) and the relative error for every single data point.
3. **Separate good from bad predictions**: using a fixed threshold on the residual, split the dataset into a "large error" and a "small error" subset, and save each one for later use.
4. **Visualize a concrete example of each**: plot, for one IL taken from each subset, the experimental and predicted CO<sub>2</sub> solubility as a function of *T* and *P*, to see what a "good" and a "bad" prediction actually look like.

</div>

#### [E.4.1] Save y_hat of the training and of the test set in the dataCO2 dataframe

In [ ]:
display(dataCO2)
df_ytrain_hat = pd.DataFrame(ytrain_hat,index=x_train.index,columns=["x_CO2_hat"])
df_ytrain_hat["set"] = "train"
#display(df_ytrain_hat)
df_ytest_hat = pd.DataFrame(ytest_hat,index=x_test.index,columns=["x_CO2_hat_tss"])
df_ytest_hat["set_tss"] = "test"
#display(df_ytest_hat)
dataCO2_hat = pd.concat([dataCO2,df_ytrain_hat,df_ytest_hat],axis=1)
dataCO2_hat.update({'x_CO2_hat': dataCO2_hat.pop('x_CO2_hat_tss')})
dataCO2_hat.update({'set':       dataCO2_hat.pop('set_tss')})
display(dataCO2_hat)

#### [E.4.2] Calculate residuals and relative errors

<div class="introE">

<table width="100%" cellpadding="6"><tr><td bgcolor="#e6e6e6"><span style="color:#000000">📖 <b>Introduction</b></span></td></tr></table>
    
They will be stored in a new `dataCO2_hat` dataframe
</div>

In [ ]:
dataCO2_hat["residual"] = dataCO2_hat.loc[:,["x_CO2","x_CO2_hat"]].apply(lambda x: x.iloc[0]-x.iloc[1], axis=1)
dataCO2_hat["relative err."] = dataCO2_hat.loc[:,["x_CO2","x_CO2_hat"]].apply(lambda x: round(100*abs(x.iloc[0]-x.iloc[1])/x.iloc[0],1), axis=1)
move = "x_CO2_hat"
column_to_move = dataCO2_hat.pop(move)
dataCO2_hat.insert(6, move, column_to_move)
move = "residual"
column_to_move = dataCO2_hat.pop(move)
dataCO2_hat.insert(7, move, column_to_move)
move = "relative err."
column_to_move = dataCO2_hat.pop(move)
dataCO2_hat.insert(8, move, column_to_move)
move = "set"
column_to_move = dataCO2_hat.pop(move)
dataCO2_hat.insert(9, move, column_to_move)
display(dataCO2_hat)

#### [E.4.3] Split the dataset into well- and poorly-predicted points

<div class="introE">

<table width="100%" cellpadding="6"><tr><td bgcolor="#e6e6e6"><span style="color:#000000">📖 <b>Introduction</b></span></td></tr></table>

Rather than ranking individual errors, the predictions are split into two complementary subsets using a fixed threshold on the absolute residual:
- <code>dataCO2_Large_Error</code>: points whose prediction is off by at least <code>threshold</code> (here 0.20) &mdash; the worst-predicted cases
- <code>dataCO2_Small_Error</code>: all the remaining, well-predicted points

Both subsets are saved as separate CSV files for later inspection. This is also where the <code>[HMIM][BF4]</code> and <code>[BMIM][BF4]</code> examples used in the 3D surface comparison above come from: the first belongs to <code>dataCO2_Large_Error</code>, the second to <code>dataCO2_Small_Error</code>.

</div>

In [ ]:
threshold = 0.20
dataCO2_Large_Error = dataCO2_hat.query("abs(residual) >= @threshold")
display(dataCO2_Large_Error)
dataCO2_Large_Error.to_csv("./DS4B-CO2-data/dataCO2_withPrediction_Wrong.csv")
dataCO2_Small_Error = dataCO2_hat.query("abs(residual) < @threshold")
display(dataCO2_Small_Error)
dataCO2_Small_Error.to_csv("./DS4B-CO2-data/dataCO2_withPrediction_Right.csv")

#### [E.4.4] Plot x_CO2 & x_CO2_hat = f(T,P) for a given IL

<div class="introE">

<table width="100%" cellpadding="6"><tr><td bgcolor="#e6e6e6"><span style="color:#000000">📖 <b>Introduction</b></span></td></tr></table>

For a chosen IL, the experimental (<code>x_CO2</code>) and predicted (<code>x_CO2_hat</code>) solubilities are plotted as two 3D surfaces as a function of *T* and *P*. Since the experimental measurements are not located on a regular (T,P) grid, <code>scipy.interpolate.griddata</code> is used to interpolate them onto one before plotting.

<b>Note:</b> the interpolation method is <code>'linear'</code>, not <code>'cubic'</code>. Cubic interpolation looks smoother, but it is an <i>exact</i> interpolant built from local derivative estimates: on sparse, unevenly spaced experimental data it tends to overshoot well beyond the physical range of the data (checked on this very dataset: up to 0.78 for a quantity that never exceeds 0.65!). Linear interpolation stays bounded by the convex hull of the data &mdash; a safer, more honest default when you don't know in advance how trustworthy the interpolation will be.

The red markers show the actual experimental points, so you can judge for yourself how much of the surface is genuine signal versus interpolation guesswork in poorly sampled regions.

</div>


In [ ]:
import plotly.graph_objects as go
from scipy.interpolate import griddata

TN, PN, xCO2N, xCO2hatN = 'T (K)', 'P (bar)', 'x_CO2', 'x_CO2_hat'
ILN = '[HMIM][BF4]'   # example with a large prediction error

sub = dataCO2_hat.query("IL == @ILN")

Ti = np.linspace(sub[TN].min(), sub[TN].max(), 100)
Pi = np.linspace(sub[PN].min(), sub[PN].max(), 100)
T_grid, P_grid = np.meshgrid(Ti, Pi)

xCO2_grid     = griddata((sub[TN], sub[PN]), sub[xCO2N],    (T_grid, P_grid), method='linear')
xCO2_hat_grid = griddata((sub[TN], sub[PN]), sub[xCO2hatN], (T_grid, P_grid), method='linear')

fig = go.Figure(data=[
    go.Surface(
        x=T_grid, y=P_grid, z=xCO2_grid,
        colorscale='viridis', showscale=False, opacity=0.85,
        name='experimental (interpolated)', showlegend=True,
    ),
    go.Surface(
        x=T_grid, y=P_grid, z=xCO2_hat_grid,
        colorscale='greys', showscale=False, opacity=0.85,
        name='predicted (interpolated)', showlegend=True,
    ),
    go.Scatter3d(
        x=sub[TN], y=sub[PN], z=sub[xCO2N],
        mode='markers', marker=dict(size=4, color='red'),
        name='experimental data points',
    ),
])

fig.update_layout(
    title=f'CO2 solubility — {ILN}',
    autosize=False, width=1000, height=900,
    margin=dict(l=65, r=50, b=65, t=90),
    legend=dict(x=0.02, y=0.98),
    scene=dict(xaxis_title=TN, yaxis_title=PN, zaxis_title=xCO2N),
)
fig.show()

## [F] Model interpretability, aka eXplainable AI: SHAP analysis<a id="shap"></a>

<figure>
<img width="50%" src="./DS4B-iris-images/DALL·E 2025-02-17 21.08.04 - A futuristic digital artwork featuring Rodin's 'The Thinker' statue integrated with Explainable AI (XAI). The statue is surrounded by a glowing neural.webp" style="display:block; margin-left:auto; margin-right:auto" id="ThinkerByRodin"/>  
<figcaption>Image generated with ChatGPT 5.0</figcaption>
</figure>

<div class="introE">

<table width="100%" cellpadding="6"><tr><td bgcolor="#e6e6e6"><span style="color:#000000">📖 <b>Introduction</b></span></td></tr></table>

Error metrics (MAE, R&sup2;) and residuals tell us <i>how much</i> the model is wrong, but not <i>why</i> it predicts what it predicts. <a href="https://shap.readthedocs.io/">SHAP</a> (SHapley Additive exPlanations) assigns to each feature, for each individual prediction, a contribution value rooted in cooperative game theory: how much that feature pushed the prediction above or below the average prediction over the training set.

SHAP stands for **SH**apley **A**dditive ex**P**lanations. [Shapley values](https://en.wikipedia.org/wiki/Shapley_value) are a way to explain the output of any machine learning model. It uses a game theoretic approach that measures each *player*'s contribution to the final outcome. One of the fundamental properties of Shapley values is that they always sum up to the difference between the game outcome when all players are present and the game outcome when no players are present.

The Shapley value was named in honor of **Lloyd Shapley**, who introduced it in 1951 and won the **Nobel Memorial Prize in Economic Sciences** for it in 2012.

<figure>
<img width = "500px" src="./ML-Figures/ImagesDivers/Shapley-NobelPrize.jpg" alt="Nobel Shapley" />
<figcaption>Lloyd Shapley receiving the Nobel Prize in Economic Sciences, 2012, for "<i>the theory of stable allocations and the practice of market design</i>"</figcaption>
</figure> 
<br>

In a SHAP analyzis, each **feature** is assigned an **importance value** representing its **contribution to the model's output**.

SHAP values show **how each** feature - or **descriptor** - **affects <u>each</u> final prediction**, the significance of each feature compared to others, and the model's reliance on the interaction between features

The ANN is treated here as a black box (only `ANNmodel.predict` is used), so `shap.KernelExplainer` is the natural, model-agnostic choice. It is also the slowest one, so to keep the computation tractable:
- the **background** distribution (the reference point against which contributions are measured) is summarized to 50 representative points with `shap.kmeans`, instead of the full training set
- only a **random subsample** of the test set is explained, instead of all ~1900 points

</div>

### [F.1] Setting up the explainer

<div class="introE">

<table width="100%" cellpadding="6"><tr><td bgcolor="#e6e6e6"><span style="color:#000000">📖 <b>Introduction</b></span></td></tr></table>

<code>shap.KernelExplainer</code> needs a reference ("background") distribution and a set of points to explain. Both are kept small on purpose, since this explainer re-queries the model many times per prediction.

</div>

In [ ]:
import shap
shap.initjs()

background = shap.kmeans(x_train, 50)

n_explain = 200
x_test_sample = x_test.sample(n=min(n_explain, len(x_test)), random_state=0)

predict_fn = lambda data: ANNmodel.predict(data, verbose=0).flatten()
explainer = shap.KernelExplainer(predict_fn, background)
shap_values = explainer.shap_values(x_test_sample, nsamples=100)

### [F.2] Global feature importance

<div class="introE">

<table width="100%" cellpadding="6"><tr><td bgcolor="#e6e6e6"><span style="color:#000000">📖 <b>Introduction</b></span></td></tr></table>

Two complementary views of the same SHAP values: the **bar chart** ranks features by their average <i>magnitude</i> of impact, while the **beeswarm plot** below it additionally shows the <i>direction</i> of each effect (does a high/low feature value push the prediction up or down?) and how that effect varies across the explained points.

</div>

In [ ]:
t4pPC.centerTitle("Bar chart")
shap.summary_plot(shap_values, x_test_sample, plot_type="bar", show=True)

t4pPC.centerTitle("Beeswarm plot")
shap.summary_plot(shap_values, x_test_sample, show=True)

### [F.3] A closer look: the Tf2N anion

<div class="introE">

<table width="100%" cellpadding="6"><tr><td bgcolor="#e6e6e6"><span style="color:#000000">📖 <b>Introduction</b></span></td></tr></table>

If everything went right, <code>[Tf2N]</code> one-hot column comes out near the top of the ranking above. Before investigating *why*, here is the anion itself: bis(trifluoromethanesulfonyl)imide, [N(SO<sub>2</sub>CF<sub>3</sub>)<sub>2</sub>]<sup>&minus;</sup>.

</div>

In [ ]:
from rdkit import Chem
from rdkit.Chem.Draw import rdMolDraw2D
from IPython.display import SVG

molTf2N = Chem.MolFromSmiles("C(F)(F)(F)S(=O)(=O)[N-]S(=O)(=O)C(F)(F)F")
d = rdMolDraw2D.MolDraw2DSVG(350, 250)
d.DrawMolecule(molTf2N)
d.FinishDrawing()
svg = d.GetDrawingText()
display(SVG(svg))

### [F.4] Does Tf2N really favour CO2 solubility, independently of the model?

<div class="introE">

<table width="100%" cellpadding="6"><tr><td bgcolor="#e6e6e6"><span style="color:#000000">📖 <b>Introduction</b></span></td></tr></table>

SHAP only explains what the <i>model</i> learned. It is not, by itself, proof of a real physical effect. The plot below checks the claim directly in the raw experimental data, binning by pressure (the single most influential feature) so that the comparison between <code>[Tf2N]</code> and the other anions isn't confounded by it.

</div>

In [ ]:
df_anion = dataCO2.copy()
df_anion["has_Tf2N"] = df_anion["[Tf2N]"] == 1

bins = [0, 5, 10, 20, 40, 80, 160, 320, 1000]
df_anion["P_bin"] = pd.cut(df_anion["P (bar)"], bins)

fig, ax = plt.subplots(figsize=(10, 6))
sns.pointplot(
    data=df_anion, x="P_bin", y="x_CO2",
    hue="has_Tf2N", hue_order=[False, True],
    errorbar="sd",  # show the spread (+/- 1 std) of the raw x_CO2 values in each group,
                    # not a 95% CI on the mean -- the point here is the real chemical
                    # variability within a pressure bin, not the precision of the estimate
    ax=ax,
)

# re-use the handles sns.pointplot already created (correct colors),
# only replace the text labels - never let ax.legend() re-derive handles on its own
handles, _ = ax.get_legend_handles_labels()
ax.legend(handles=handles, labels=["other anions", "Tf2N"], title="anion")

ax.set_xlabel("P / bar", fontsize=14)
ax.set_ylabel("$x_\mathrm{CO_2}$", fontsize=14)
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

### [F.5] Local explanations: a well- vs. a poorly-predicted example

<div class="introE">

<table width="100%" cellpadding="6"><tr><td bgcolor="#e6e6e6"><span style="color:#000000">📖 <b>Introduction</b></span></td></tr></table>

Global importance says what matters <i>on average</i>; it says nothing about any single prediction. The **force plots** below contrast one point from <code>dataCO2_Large_Error</code> with one from <code>dataCO2_Small_Error</code>, showing which features pushed each individual prediction away from (or back towards) the experimental value.

</div>

In [ ]:
X_all = pd.concat([x_train, x_test])
idx_large = dataCO2_Large_Error.index[0]
idx_small = dataCO2_Small_Error.index[0]
x_examples = X_all.loc[[idx_large, idx_small]]

shap_examples = explainer.shap_values(x_examples, nsamples=100)

for i, (idx, label) in enumerate(zip(x_examples.index, ["large error", "small error"])):
    il_name = dataCO2_hat.loc[idx, "IL"]
    x_exp   = dataCO2_hat.loc[idx, "x_CO2"]
    x_pred  = dataCO2_hat.loc[idx, "x_CO2_hat"]
    T_exp   = dataCO2_hat.loc[idx, "T (K)"]
    P_exp   = dataCO2_hat.loc[idx, "P (bar)"]
    print(f"--- {label} example (IL: {il_name}) ---")
    print(f"{color.BOLD}{color.RED}experimental x_CO2 = {x_exp:.3f}{color.OFF}"
          f"   (predicted = {x_pred:.3f})   at T = {T_exp:.1f} K, P = {P_exp:.1f} bar")
    shap.force_plot(explainer.expected_value, shap_examples[i], x_examples.iloc[i], matplotlib=True)

## [G] Take-home message

<div class="introE">

<table width="100%" cellpadding="6"><tr><td bgcolor="#e6e6e6"><span style="color:#000000">📖 <b>Introduction</b></span></td></tr></table>

This tutorial is not really about CO<sub>2</sub> solubility, or even about neural networks &mdash; it is about **the habits of mind an ML practitioner needs**, illustrated on data and a model that look fine at first glance:

- **A widely cited, published result is not automatically trustworthy.** Reproducing Song *et al.*'s approach step by step exposed real, checkable issues: unconsolidated data, and a collapsed model hidden behind seemingly decent-looking metrics.
- **Always audit your data before modeling it.** 433 exact duplicate rows and 151 contradictory (IL, T, P) measurements were sitting in a dataset that "looked" clean &mdash; they would silently bias any train/test split.
- **Unscaled features can silently break gradient-based learning.** Mixing 0/1 descriptors with raw *T* and *P* made Model 1 collapse to predicting roughly the mean solubility &mdash; while its MAE and (mislabeled) R&sup2; still looked almost reasonable.
- **Global feature importance (SHAP) explains the model, not nature.** `[Tf2N]`'s high ranking is a real chemical effect, but it was necessary to check whether it could have been related to its over-representation in the dataset.
- **Interpretability tools are a starting point for scientific questions, not an endpoint.** The most useful SHAP finding here wasn't a number, but the follow-up question it prompted: *is this really true, independently of this particular model?*
- **Interpretability can reassure as well as warn.** P and T coming out as the dominant, physically expected drivers (solubility rising with pressure, falling with temperature) is itself meaningful: it shows the network learned real physical chemistry rather than a dataset-specific artifact &mdash; the same tool that exposed the Tf2N ambiguity also validates the parts of the model that *do* make sense.

</div>

## [H] "Simple" exercise

<div class="exE" title="Optimize the hyperparameters of Model 2 by hand">

<table width="100%" cellpadding="6"><tr><td bgcolor="#d8edf4"><span style="color:#0D47A1">📝 <b>Exercise [H.1]</b></span></td></tr></table>

Return to the <a href="#model2">corresponding section</a> and change some hyperparameters. Carefully note the accuracy of each attempt, and watch out for a possible overfitting. Suggestions:
- Try to increase the number of neurons of the hidden layer
- Replace the `tanh` activation function with the `relu` activation function
- Add a second hidden layer (this is **deep learning**!)

Do you find a better model than the original model 2?
</div>

In [ ]:
## Enter your code here


## [I] Short bibliography

<div class="introE">

<table width="100%" cellpadding="6"><tr><td bgcolor="#e6e6e6"><span style="color:#000000">📖 <b>Introduction</b></span></td></tr></table>

- V. Venkatraman & B. K. Alsberg (**2017**), Predicting CO<sub>2</sub> capture of ionic liquids using machine learning, [*J. CO2 Util.* **21**: 162&ndash;168](https://doi.org/10.1016/j.jcou.2017.06.012)
- Z. Song, H. Shi, X. Zhang & T. Zhou (**2020**), Prediction of CO<sub>2</sub> solubility in ionic liquids using machine learning methods, [*Chem. Eng. Sci.* **223**: 115752](https://doi.org/10.1016/j.ces.2020.115752)
- X. Zheng, K. Fukuhara, Y. Hijikata, J. Pirillo, H. Sato, K. Takahashi, S. Noro & T. Nakamura (**2020**), Understanding the interactions between the bis(trifluoromethylsulfonyl)imide anion and absorbed CO<sub>2</sub> using X-ray diffraction analysis of a soft crystal surrogate, [*Commun. Chem.* **3**: 143](https://doi.org/10.1038/s42004-020-00390-1)
- L. Toucouère & R. Poteau (**2026**), Comprehensive compilation and curation of CO<sub>2</sub> solubility in 438 distinct ionic liquids: Expanding the chemical space, [*Chem. Data Collect.* **64**: 101252](https://www.sciencedirect.com/science/article/pii/S2405830026000315)
- L. Toucouère, Q. Jéré, S. Christodoulou, J. Volkman, L. Maron, F. Jolibois & R. Poteau (**2026**), Machine Learning for CO<sub>2</sub> Capture in Ionic Liquids: Generalization, Interpretability, and Screening, *submitted to ChemSusChem*
</div>

## [J] Appendix: activation functions

In [ ]:
def derivative(f, z, eps=0.000001):
    return (f(z + eps) - f(z - eps))/(2 * eps)

z = np.linspace(-5, 5, 200)
pw=4
ph=4


### [J.1] relu function

$$R(z)=\begin{cases}
\begin{array}{cc}
z & \,\,\,\,\,\,\,z>0\\
0 & \,\,\,\,\,\,\,z\leq0
\end{array}\end{cases}$$

In [ ]:
def relu(z):
    return np.maximum(0, z)

figR, axR = plt.subplots(1, 1)
figR.set_size_inches(pw,ph)
axR.axhline(y=0, linewidth=2, linestyle='--', color='lightgray')
axR.axvline(x=0, linewidth=2, linestyle='--', color='lightgray')
axR.plot(z, relu(z),linewidth=4,label="ReLU")
axR.plot(z, derivative(relu, z), linewidth=4, alpha=0.6, label="dReLU/dx")
axR.legend(loc='upper left', shadow=True, fontsize='14')
axR.set_xlabel('x',fontdict={'fontsize':16})
axR.set_ylabel('f(x)',fontdict={'fontsize':16})
axR.set_xlim(-5, 5)
axR.set_ylim(-0.5, 7)
axR.tick_params(labelsize = 14)
plt.show()


### [J.2] tanh function

In [ ]:
def tanh(z):
    return np.tanh(z)

figT, axT = plt.subplots(1, 1)
figT.set_size_inches(pw,ph)
axT.axhline(y=0, linewidth=2, linestyle='--', color='lightgray')
axT.axvline(x=0, linewidth=2, linestyle='--', color='lightgray')
axT.plot(z, tanh(z),linewidth=4,label="tanh")
axT.plot(z, derivative(tanh, z), linewidth=4, alpha=0.6, label="dtanh/dx")
axT.legend(loc='upper left', shadow=True, fontsize='14')
axT.set_xlabel('x',fontdict={'fontsize':16})
axT.set_ylabel('f(x)',fontdict={'fontsize':16})
axT.set_xlim(-5, 5)
axT.set_ylim(-2, 2)
axT.tick_params(labelsize = 14)
plt.show()


### [J.3] softmax function
$$\sigma_{i}(\boldsymbol{x})=\frac{e^{x_{i}}}{\sum_{j=1}^{K}e^{x_{j}}}$$

In [ ]:
def softmax(x):
    e_x = np.exp(x - np.max(x)) #this implentation has a better numerical stability
    return e_x / e_x.sum()

z2 = [-2., -1., -0.5, 0., 0.5, 1., 2.]
figS, axS = plt.subplots(1, 1)
figS.set_size_inches(pw,ph)
axS.axhline(y=0, linewidth=2, linestyle='--', color='lightgray')
axS.axvline(x=0, linewidth=2, linestyle='--', color='lightgray')
axS.plot(z2, softmax(z2),linewidth=4,marker='o',label="softmax")
axS.legend(loc='upper left', shadow=True, fontsize='14')
axS.set_xlabel('x',fontdict={'fontsize':16})
axS.set_ylabel('f(x)',fontdict={'fontsize':16})
axS.set_xlim(-2.1, 2.1)
axS.set_ylim(0., 1.0)
axS.tick_params(labelsize = 14)
plt.show()
print(f"Sum of softmax values: {softmax(z2).sum()}")

In [ ]:
t4pPC.end()

# END